# 09 Beta Diversity and Diet

We test whether fungal community composition is associated with `diet_type_sample`. The primary analysis uses 16-mer Bray–Curtis distances, and the sensitivity check uses whole-ASV Bray–Curtis distances. We also run an exploratory analysis using 16-mer Aitchison distances to see whether the results depend on the distance metric.

We start with the full 150-sample distance matrices from notebook 07 and the cleaned metadata from notebook 06. All analyses use the same 145 samples with a reported diet label.

In [6]:
import pandas as pd
import qiime2 as q2
from pathlib import Path
from skbio import DistanceMatrix
from qiime2 import Visualization

In [2]:
# Bray-curtis distance matrix
kmer16_distance_path = "../data/processed_ignored/07_beta_diversity/kmer16_bray_curtis_distance.qza"
asv_distance_path = "../data/processed_ignored/07_beta_diversity/asv_bray_curtis_distance.qza"
metadata_path = "../data/processed_ignored/06_diversity/Final_metadata_with_GHSL.tsv"

out_dir = "../data/processed_ignored/09_beta_diversity_diet"
vis_dir = "../data/processed/09_beta_diversity_diet"

diet_metadata_path = f"{out_dir}/diet_metadata.tsv"
kmer16_diet_distance_path = f"{out_dir}/kmer16_bray_curtis_diet.qza"
asv_diet_distance_path = f"{out_dir}/asv_bray_curtis_diet.qza"

kmer16_permanova_path = f"{vis_dir}/kmer16_bray_curtis_diet_permanova.qzv"
asv_permanova_path = f"{vis_dir}/asv_bray_curtis_diet_permanova.qzv"

kmer16_pairwise_permanova_path = f"{vis_dir}/kmer16_bray_curtis_diet_pairwise_permanova.qzv"
asv_pairwise_permanova_path = f"{vis_dir}/asv_bray_curtis_diet_pairwise_permanova.qzv"

# Additional exploratory analysis: Aitchison distance matrix
kmer16_aitchison_distance_path = "../data/processed_ignored/07_beta_diversity/kmer16_aitchison_distance.qza"
kmer16_aitchison_diet_distance_path = f"{out_dir}/kmer16_aitchison_diet.qza"
kmer16_aitchison_permanova_path = f"{vis_dir}/kmer16_aitchison_diet_permanova.qzv"
kmer16_aitchison_pairwise_permanova_path = f"{vis_dir}/kmer16_aitchison_diet_pairwise_permanova.qzv"


! mkdir -p "$out_dir" "$vis_dir"

## 1. Load the metadata and diversity artifacts

### 1.1 Metadata

Use pandas to summarize `diet_type_sample`, including missing values, for the samples in the full Bray–Curtis matrix.

In [3]:
metadata = q2.Metadata.load(metadata_path).to_dataframe()
distance_matrix = q2.Artifact.load(kmer16_distance_path).view(DistanceMatrix)

diet = metadata.reindex(list(distance_matrix.ids))["diet_type_sample"]
diet = diet.str.strip().replace("", pd.NA)
diet.value_counts(dropna=False)

diet_type_sample
Omnivore                            118
Vegetarian but eat seafood            8
Omnivore but do not eat red meat      8
Vegetarian                            6
Vegan                                 5
NaN                                   5
Name: count, dtype: int64

In [7]:
diet_metadata = diet.dropna().to_frame()
diet_sample_ids = diet_metadata.index.tolist()

### 1.2 Prepare the diversity artifacts

Subset both Bray–Curtis distance matrices to the same 145 samples with a reported diet label. Save the filtered matrices and the diet metadata for PERMANOVA.

In [8]:
q2.Metadata(diet_metadata).save(diet_metadata_path)

kmer16_diet = distance_matrix.filter(diet_sample_ids)
q2.Artifact.import_data("DistanceMatrix", kmer16_diet).save(kmer16_diet_distance_path)

asv_matrix = q2.Artifact.load(asv_distance_path).view(DistanceMatrix)
asv_diet = asv_matrix.filter(diet_sample_ids)
q2.Artifact.import_data("DistanceMatrix", asv_diet).save(asv_diet_distance_path)

'../data/processed_ignored/09_beta_diversity_diet/asv_bray_curtis_diet.qza'

## 2. PERMANOVA by diet type

### 2.1 Overall PERMANOVA on 16-mer Bray–Curtis distances

Use overall PERMANOVA with 999 permutations using the filtered 16-mer Bray–Curtis distance matrix for the 145 samples with a diet label.

In [9]:
! qiime diversity beta-group-significance \
    --i-distance-matrix "$kmer16_diet_distance_path" \
    --m-metadata-file "$diet_metadata_path" \
    --m-metadata-column diet_type_sample \
    --o-visualization "$kmer16_permanova_path"

Saved Visualization to: ../data/processed/09_beta_diversity_diet/kmer16_bray_curtis_diet_permanova.qzv


In [ ]:
Visualization.load(kmer16_permanova_path)

### 2.2 Overall PERMANOVA on whole-ASV Bray–Curtis distances

Repeat the overall diet test with 999 permutations using whole-ASV Bray–Curtis distances. Use the same 145 samples and five diet groups as in the 16-mer test.

In [10]:
! qiime diversity beta-group-significance \
    --i-distance-matrix "$asv_diet_distance_path" \
    --m-metadata-file "$diet_metadata_path" \
    --m-metadata-column diet_type_sample \
    --o-visualization "$asv_permanova_path"

Saved Visualization to: ../data/processed/09_beta_diversity_diet/asv_bray_curtis_diet_permanova.qzv


In [ ]:
Visualization.load(asv_permanova_path)

### 2.3 Compare the overall diet PERMANOVA results

| Bray–Curtis feature representation | Samples | Pseudo-F | p-value |
|---|---:|---:|---:|
| 16-mers | 145 | 0.608 | 0.989 |
| Whole ASVs | 145 | 0.778 | 0.9 |

**Results**

Neither overall PERMANOVA detected a statistically significant association between reported diet type and fungal community composition at α = 0.05. The conclusion was the same for 16-mer Bray–Curtis distances (pseudo-F = 0.608, p = 0.989) and whole-ASV Bray–Curtis distances (pseudo-F = 0.778, p = 0.900), using the same 145 samples across five diet groups.

### 2.4 Pairwise PERMANOVA on 16-mer Bray–Curtis distances

Compare the five diet groups two at a time. The overall 16-mer test gave p = 0.989, so these follow-up comparisons are exploratory. Use the adjusted q-values to interpret the ten pairwise tests.

In [12]:
! qiime diversity beta-group-significance \
    --i-distance-matrix "$kmer16_diet_distance_path" \
    --m-metadata-file "$diet_metadata_path" \
    --m-metadata-column diet_type_sample \
    --p-pairwise \
    --o-visualization "$kmer16_pairwise_permanova_path"

Saved Visualization to: ../data/processed/09_beta_diversity_diet/kmer16_bray_curtis_diet_pairwise_permanova.qzv


In [ ]:
Visualization.load(kmer16_pairwise_permanova_path)

### 2.5 Pairwise PERMANOVA on whole-ASV Bray–Curtis distances

Repeat the ten pairwise diet comparisons using whole-ASV Bray–Curtis distances. The overall whole-ASV test gave p = 0.900, so these comparisons are exploratory. Use the q-values to account for multiple testing.

In [14]:
! qiime diversity beta-group-significance \
    --i-distance-matrix "$asv_diet_distance_path" \
    --m-metadata-file "$diet_metadata_path" \
    --m-metadata-column diet_type_sample \
    --p-pairwise \
    --o-visualization "$asv_pairwise_permanova_path"

Saved Visualization to: ../data/processed/09_beta_diversity_diet/asv_bray_curtis_diet_pairwise_permanova.qzv


In [ ]:
Visualization.load(asv_pairwise_permanova_path)

### 2.6 Compare the pairwise diet PERMANOVA results

| Diet pair | 16-mer q-value | Whole-ASV q-value |
|---|---:|---:|
| Omnivore vs. Omnivore but do not eat red meat | 0.992 | 0.973 |
| Omnivore vs. Vegan | 0.992 | 0.973 |
| Omnivore vs. Vegetarian | 0.992 | 0.973 |
| Omnivore vs. Vegetarian but eat seafood | 0.992 | 0.973 |
| Omnivore but do not eat red meat vs. Vegan | 0.992 | 0.973 |
| Omnivore but do not eat red meat vs. Vegetarian | 0.992 | 0.973 |
| Omnivore but do not eat red meat vs. Vegetarian but eat seafood | 0.992 | 0.973 |
| Vegan vs. Vegetarian | 0.992 | 0.973 |
| Vegan vs. Vegetarian but eat seafood | 0.992 | 0.973 |
| Vegetarian vs. Vegetarian but eat seafood | 0.992 | 0.973 |

**Results**

After correction for multiple comparisons, none of the ten exploratory diet comparisons met the q < 0.05 threshold with either feature representation.

## 3. Exploratory PERMANOVA using Aitchison distances

The overall Bray–Curtis PERMANOVA found little evidence of an association between reported diet type and fungal community composition. We repeat the 16-mer analysis with Aitchison distances to see whether that conclusion depends on how we measure differences between samples.

Bray–Curtis measures differences in feature abundances, while Aitchison measures differences in the ratios between features. We use the same 145 samples, five diet groups, and 16-mer features so that the distance metric is the only change. Because we added this comparison after seeing the Bray–Curtis results, we interpret it as exploratory.

### 3.2 Prepare the 16-mer Aitchison distance matrix

Subset the Aitchison distance matrix from notebook 07 to the same 145 samples used in the Bray–Curtis diet analysis.

In [18]:
kmer16_aitchison = q2.Artifact.load(kmer16_aitchison_distance_path).view(DistanceMatrix)
kmer16_aitchison_diet = kmer16_aitchison.filter(diet_sample_ids)

q2.Artifact.import_data("DistanceMatrix", kmer16_aitchison_diet).save(
    kmer16_aitchison_diet_distance_path
)

'../data/processed_ignored/09_beta_diversity_diet/kmer16_aitchison_diet.qza'

### 3.3 Overall PERMANOVA on 16-mer Aitchison distances

Run one exploratory overall PERMANOVA with 999 permutations using the same 145 samples and five diet groups as in the 16-mer Bray–Curtis test.

In [20]:
! qiime diversity beta-group-significance \
    --i-distance-matrix "$kmer16_aitchison_diet_distance_path" \
    --m-metadata-file "$diet_metadata_path" \
    --m-metadata-column diet_type_sample \
    --o-visualization "$kmer16_aitchison_permanova_path"

Saved Visualization to: ../data/processed/09_beta_diversity_diet/kmer16_aitchison_diet_permanova.qzv


In [ ]:
Visualization.load(kmer16_aitchison_permanova_path)

### 3.4 Compare the overall 16-mer PERMANOVA results

| Distance metric | Samples | Pseudo-F | p-value |
|---|---:|---:|---:|
| Bray–Curtis | 145 | 0.608 | 0.989 |
| Aitchison | 145 | 1.036 | 0.371 |

**Results**

Using the same 145 samples and five diet groups, the exploratory Aitchison PERMANOVA yielded a lower p-value than the Bray–Curtis test (0.371 vs. 0.989). However, both p-values exceeded 0.05, so the overall conclusion was consistent across the two distance metrics.

### 3.5 Pairwise PERMANOVA on 16-mer Aitchison distances

Compare the five diet groups two at a time using 16-mer Aitchison distances. Use the q-values to account for multiple testing across the ten exploratory comparisons.

In [7]:
print("Notebook directory:", Path.cwd())
print("Metadata path:", repr(diet_metadata_path))
print("Metadata file exists:", Path(diet_metadata_path).is_file())

Notebook directory: /home/jovyan/FunGut/scripts
Metadata path: '../data/processed_ignored/09_beta_diversity_diet/diet_metadata.tsv'
Metadata file exists: True


In [9]:
! qiime diversity beta-group-significance \
    --i-distance-matrix "{kmer16_aitchison_diet_distance_path}" \
    --m-metadata-file "{diet_metadata_path}" \
    --m-metadata-column diet_type_sample \
    --p-pairwise \
    --o-visualization "{kmer16_aitchison_pairwise_permanova_path}"

Saved Visualization to: ../data/processed/09_beta_diversity_diet/kmer16_aitchison_diet_pairwise_permanova.qzv


In [ ]:
Visualization.load(kmer16_aitchison_pairwise_permanova_path)

### 3.6 Compare the pairwise 16-mer PERMANOVA results

| Diet pair | Bray–Curtis q-value | Aitchison q-value |
|---|---:|---:|
| Omnivore vs. Omnivore but do not eat red meat | 0.992 | 0.974 |
| Omnivore vs. Vegan | 0.992 | 0.060 |
| Omnivore vs. Vegetarian | 0.992 | 0.948 |
| Omnivore vs. Vegetarian but eat seafood | 0.992 | 0.948 |
| Omnivore but do not eat red meat vs. Vegan | 0.992 | 0.060 |
| Omnivore but do not eat red meat vs. Vegetarian | 0.992 | 0.948 |
| Omnivore but do not eat red meat vs. Vegetarian but eat seafood | 0.992 | 0.948 |
| Vegan vs. Vegetarian | 0.992 | 0.163 |
| Vegan vs. Vegetarian but eat seafood | 0.992 | 0.948 |
| Vegetarian vs. Vegetarian but eat seafood | 0.992 | 0.948 |

**Results**

None of the ten exploratory diet comparisons met the q < 0.05 threshold with either distance metric. With Aitchison distances, the two comparisons of Vegan samples with the omnivore groups had the lowest adjusted q-values (both q = 0.060). Both comparisons included the same five Vegan samples, and the overall Aitchison test gave p = 0.371.

## 4. Discussion

The overall Bray–Curtis tests did not detect a significant association between reported diet type and fungal community composition. The conclusion was the same for 16-mers and whole ASVs, and the exploratory 16-mer Aitchison analysis gave a similar overall result. This is consistent with the diet-colored PCoA plots in notebook 07, where the diet groups did not clearly separate along the first two axes.

With Aitchison distances, the two comparisons between Vegan samples and the two omnivore groups had the lowest adjusted q-values (q = 0.060 for both). Neither comparison met the 0.05 threshold, and both involved the same five Vegan samples. The diet groups were uneven in size, with 118 samples in the Omnivore group and only 5–8 in each of the other four groups. This may have made differences harder to detect, so the results leave open whether diet-related differences would be detectable in a larger, more balanced dataset.